# Datamine AED Process Example

This notebook demonstrates how to configure and run the **`aed`** process wrapper in `dmstudio`.

## Process Description

## AED

File editor for creation, validation and modification of files.

Files may be set-up or validated using an optional definition file. This file may itself first be set-up in **AED** , or an existing definition file may be used, initially modified by the editor if required. Data may be entered by typing in the individual records or existing system files may be read in. In addition, character format files may be output. Any existing database file may be entered, and records may be inserted, modified or deleted. New records may be added under control of validation criteria set up in the definition file.

AED operates by displaying on the screen a series of 'pads', which provide a 'view' of part of the file. Each 'pad' consists of an 'information' line, a line displaying the field names, a data area of 14 records, and an Input and a Message line. Cursor movement is controlled by the usual cursor control keys or by entry of special characters on the input line. Extensive editing functions are provided by a set of sub-commands which are also entered on the input line. These sub-commands are described below.

**Note** : The Datamine **Table Editor** offers a more interactive approach to modifying Datamine (,dm) binary files.

### Input Files:

* **in** (Input file to be edited. If this file does not exist, it will be created: first a definition file will be created (named "name".%) into which you enter the required fields, validation info, etc. Then the definition file is written out and used to define the IN file, into which you enter data.):
  Overwritten
  Required=No

* **defn** (Definition file. Must contain the following fields:- FIELD, TYPE, LENGTH, STORED, FORM, FMIN, FMAX, INCR, VALUES. If neither the input file nor the definition file exist, both will be created. If the input file does not exist, but a definition file is entered, then the input file will be created from the definition file. Whenever a definition file exists, it may be used for validation. If both the input and the validation file exist, then input file entries will be validated against the definition file.):
  Overwritten
  Required=No

* **apphlp** (Undefined):
  Input file containing application specific help information that will be included in
  the help display. An alphanumeric field named APPHLP will be expected, with a maximum of
  56 characters displayed.
  Required=No

### Output Files:

* **out** (Table):
  Edited output file. If this is not specified, then the input file will be overwritten.
  Required=No

### Fields:

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('aed')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute aed
print("Running aed...")
dm_cmd.aed(
    # in_i='t_assays',  # optional
    # defn_i='optional',  # optional
    # apphlp_i='optional',  # optional
    # out_o='t_aed_out',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("aed execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_aed_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")